In [12]:
import os
os.environ["MODEL_KEY"]="ollama"
from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel

In [13]:
local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    )
)

In [14]:

agent = Agent(
    model=local_model,
    name="Greeter",                                    
    instructions="You are a friendly assistant. Be concise.",  
   )
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)

The capital of Pakistan is Islamabad.


In [15]:
from agents import Agent, Runner

agent = Agent(name="Analyst", instructions="You analyze topics briefly.", model=local_model )

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

OPENAI_API_KEY is not set, skipping trace export


Final Output:  Python is popular for AI due to five key factors:

*   **Rich Ecosystem:** Libraries like TensorFlow...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


In [17]:
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Turn 1: A **list** is a built-in Python data structure used to store an ordered collection of items.

**Key Characteristics:**
*   **Ordered:** Items have a defined index (starting at 0).
*   **Mutable:** You can change, add, or remove items after creation.
*   **Heterogeneous:** Can contain mixed data types (ints, strings, etc.).
*   **Syntax:** Defined using square brackets `[]`.

**Example:**
```python
# Creating a list
items = [1, "hello", 3.14]

# Accessing and modifying
items[0] = 2          # Change value
items.append("world") # Add value
```



OPENAI_API_KEY is not set, skipping trace export


Turn 2: There are two primary ways to sort a list:

1.  **`.sort()`**: Modifies the original list (**in-place**).
2.  **`sorted()`**: Returns a **new list**, leaving the original unchanged.

```python
nums = [3, 1, 2]

# Method 1: In-place
nums.sort()
# nums is now [1, 2, 3]

# Method 2: New list
nums = [3, 1, 2]
new_nums = sorted(nums)
# nums is still [3, 1, 2]
# new_nums is [1, 2, 3]
```

**Tip:** Add `reverse=True` to either method for descending order (e.g., `nums.sort(reverse=True)`).



OPENAI_API_KEY is not set, skipping trace export


Turn 3: You use the argument `reverse=True` with either method.

```python
nums = [3, 1, 2]

# In-place
nums.sort(reverse=True)
# nums becomes [3, 2, 1]

# New list
nums = [3, 1, 2]
new_nums = sorted(nums, reverse=True)
# new_nums is [3, 2, 1]
```


OPENAI_API_KEY is not set, skipping trace export


In [19]:
from agents import Agent, Runner

agent = Agent(name="Streamer", instructions="Explain briefly.",model=local_model)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Machine Learning (ML).
    *   Constraint: Explain in exactly 2 sentences.
    *   Tone: Brief, clear, informative.

2.  **Drafting - Attempt 1:**
    Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. It uses algorithms to identify patterns and make decisions or predictions based on new information.

3.  **Review - Attempt 1:**
    *   Sentence 1: Defines ML and its relation to AI/programming.
    *   Sentence 2: Explains how it works (algorithms, patterns, predictions).
    *   Count: 2 sentences.
    *   Quality: Good, accurate.

4.  **Drafting - Attempt 2 (Simpler):**
    Machine learning is a technology that allows computers to improve at tasks through experience rather than direct instructions. By analyzing large amounts of data, these systems can recognize patterns and make accurate predictions on their own.

5.  **Review - Attempt 2:

OPENAI_API_KEY is not set, skipping trace export
